In [1]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ================================
# 1. CẤU HÌNH (HYPERPARAMETERS)
# ================================
SEQ_LENGTH = 60
STEP = 30
PAD_VALUE = -99.0         # Giá trị dùng để Masking cho BiLSTM
CONF_THRESHOLD = 0.3      # Ngưỡng lọc nhiễu: Confidence < 0.4 sẽ bị nội suy lại
SAVE_DIR = "/kaggle/working/processed_data"

os.makedirs(SAVE_DIR, exist_ok=True)

# ================================
# 2. QUÉT VÀ CHIA DATASET
# ================================
print("Scanning dataset...")

fall_files = glob.glob('/kaggle/input/**/Fall/Keypoints_CSV/*.csv', recursive=True)
no_fall_files = glob.glob('/kaggle/input/**/No_Fall/Keypoints_CSV/*.csv', recursive=True)

if len(fall_files) == 0 or len(no_fall_files) == 0:
    raise ValueError("Không tìm thấy file CSV! Hãy kiểm tra lại đường dẫn.")

df_data = pd.DataFrame({
    "filepath": fall_files + no_fall_files,
    "label": [1]*len(fall_files) + [0]*len(no_fall_files)
})

train_df, temp_df = train_test_split(
    df_data, test_size=0.30, stratify=df_data["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

print(f"Train: {len(train_df)} videos | Val: {len(val_df)} videos | Test: {len(test_df)} videos")

# ================================
# 3. HÀM XỬ LÝ 1 VIDEO (KHỬ NHIỄU BẰNG CONFIDENCE)
# ================================
def process_single_video(filepath):
    df = pd.read_csv(filepath)
    if df.empty:
        raise ValueError("File trống")
        
    df.columns = df.columns.str.strip()

    # ---------------------------------------------------------
    # BƯỚC QUAN TRỌNG: LỌC NHIỄU DỰA TRÊN CONFIDENCE
    # Đưa các tọa độ X, Y có độ tin cậy thấp về dạng NaN
    # ---------------------------------------------------------
    if 'Confidence' in df.columns:
        df.loc[df['Confidence'] < CONF_THRESHOLD, ['X', 'Y']] = np.nan

    # Chỉ Pivot X và Y (Bỏ Confidence đi vì đã hoàn thành nhiệm vụ lọc)
    pivot_df = df.pivot_table(
        index="Frame",
        columns="Keypoint",
        values=["X", "Y"]
    )

    # Đổi tên cột: VD: "Left Hip_X", "Left Hip_Y"
    pivot_df.columns = [f"{c[1]}_{c[0]}" for c in pivot_df.columns]
    pivot_df.sort_index(inplace=True)

    # ---------------------------------------------------------
    # NỘI SUY TUYẾN TÍNH (INTERPOLATION)
    # Đoán lại các điểm NaN một cách mượt mà dựa vào frame trước và sau
    # ---------------------------------------------------------
    pivot_df = pivot_df.interpolate(method='linear', limit_direction='both').fillna(0)
    
    cols = pivot_df.columns
    def get_col(name):
        return pivot_df[name].values if name in cols else np.zeros(len(pivot_df))

    # --- TÍNH TỌA ĐỘ TRUNG TÂM ---
    l_hip_x, l_hip_y = get_col("Left Hip_X"), get_col("Left Hip_Y")
    r_hip_x, r_hip_y = get_col("Right Hip_X"), get_col("Right Hip_Y")
    
    l_sh_x, l_sh_y = get_col("Left Shoulder_X"), get_col("Left Shoulder_Y")
    r_sh_x, r_sh_y = get_col("Right Shoulder_X"), get_col("Right Shoulder_Y")

    midhip_x, midhip_y = (l_hip_x + r_hip_x) / 2.0, (l_hip_y + r_hip_y) / 2.0
    neck_x, neck_y = (l_sh_x + r_sh_x) / 2.0, (l_sh_y + r_sh_y) / 2.0

    # Dữ liệu gốc (17 khớp x 2 (X,Y) = 34 Features)
    base_data = pivot_df.values 

    # --- TÍNH EXTRA FEATURES ---
    features = []

    # 1. Velocity (Vận tốc cho 34 cột X, Y)
    if len(base_data) > 1:
        velocity = np.diff(base_data, axis=0)
        velocity = np.vstack([velocity[0], velocity]) # Bù lại frame đầu
    else:
        velocity = np.zeros_like(base_data)
    features.append(velocity)

    # 2. Hip height (1 Feature)
    hip_height = midhip_y.reshape(-1, 1)
    features.append(hip_height)

    # 3. Body angle (1 Feature)
    dx = neck_x - midhip_x
    dy = neck_y - midhip_y
    angle = np.arctan2(dy, dx).reshape(-1, 1)
    features.append(angle)

    # Gộp tất cả: 34 (Base) + 34 (Velocity) + 1 (Hip_H) + 1 (Angle) = 70 Features
    extra_data = np.concatenate(features, axis=1)
    final_data = np.concatenate([base_data, extra_data], axis=1)

    return final_data

# ================================
# 4. HUẤN LUYỆN SCALER (TỐI ƯU RAM)
# ================================
print("Fitting scaler on train set...")
scaler = MinMaxScaler()
failed_train_files = 0

for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Fitting Scaler"):
    try:
        data = process_single_video(row["filepath"])
        scaler.partial_fit(data)
    except Exception:
        failed_train_files += 1
        continue

print(f"Scaler fitted. (Bỏ qua {failed_train_files} file lỗi/trống)")

# ================================
# 5. TẠO SEQUENCE DATASET
# ================================
def build_dataset(df_split, desc="dataset"):
    X_all, y_all = [], []
    failed_files = 0

    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc=desc):
        try:
            # 1. Đọc và tạo 70 features đã được làm sạch
            data = process_single_video(row["filepath"])

            # 2. Chuẩn hóa Data
            data = scaler.transform(data)

            # 3. Padding nếu video ngắn hơn SEQ_LENGTH (Pad bằng -99.0)
            if len(data) < SEQ_LENGTH:
                pad_len = SEQ_LENGTH - len(data)
                pad = np.full((pad_len, data.shape[1]), PAD_VALUE)
                data = np.vstack([data, pad])

            # 4. Cắt Sliding Window
            for i in range(0, len(data) - SEQ_LENGTH + 1, STEP):
                window = data[i:i + SEQ_LENGTH]
                X_all.append(window)
                y_all.append(row["label"])

        except Exception:
            failed_files += 1
            continue

    if failed_files > 0:
        print(f"[{desc}] Bỏ qua {failed_files} file lỗi/trống.")
        
    # Lưu kiểu float32 để tiết kiệm 50% RAM so với float64 mặc định
    return np.array(X_all, dtype=np.float32), np.array(y_all, dtype=np.int32)

# ================================
# 6. THỰC THI VÀ LƯU KẾT QUẢ
# ================================
print("\nBuilding datasets...")
X_train, y_train = build_dataset(train_df, "Train")
X_val, y_val = build_dataset(val_df, "Val")
X_test, y_test = build_dataset(test_df, "Test")

np.save(f"{SAVE_DIR}/X_train.npy", X_train)
np.save(f"{SAVE_DIR}/y_train.npy", y_train)
np.save(f"{SAVE_DIR}/X_val.npy", X_val)
np.save(f"{SAVE_DIR}/y_val.npy", y_val)
np.save(f"{SAVE_DIR}/X_test.npy", X_test)
np.save(f"{SAVE_DIR}/y_test.npy", y_test)

# ================================
# 7. TỔNG KẾT
# ================================
print("\n" + "="*50)
print("🚀 DATASET SẴN SÀNG CHO BiLSTM 🚀")
print("="*50)
print(f"Train: {X_train.shape} | Labels: {y_train.shape}")
print(f"Val:   {X_val.shape}  | Labels: {y_val.shape}")
print(f"Test:  {X_test.shape}  | Labels: {y_test.shape}")
print(f"Saved to: {SAVE_DIR}")
print(f"Số lượng Features: {X_train.shape[2]} (34 X,Y + 34 Vel + 1 Hip_H + 1 Angle)")
print(f"Masking padding value: {PAD_VALUE}")
print("="*50)

Scanning dataset...
Train: 4891 videos | Val: 1048 videos | Test: 1049 videos
Fitting scaler on train set...


Fitting Scaler:   0%|          | 0/4891 [00:00<?, ?it/s]

Scaler fitted. (Bỏ qua 1018 file lỗi/trống)

Building datasets...


Train:   0%|          | 0/4891 [00:00<?, ?it/s]

[Train] Bỏ qua 1018 file lỗi/trống.


Val:   0%|          | 0/1048 [00:00<?, ?it/s]

[Val] Bỏ qua 213 file lỗi/trống.


Test:   0%|          | 0/1049 [00:00<?, ?it/s]

[Test] Bỏ qua 237 file lỗi/trống.

🚀 DATASET SẴN SÀNG CHO BiLSTM 🚀
Train: (7085, 60, 70) | Labels: (7085,)
Val:   (1501, 60, 70)  | Labels: (1501,)
Test:  (1487, 60, 70)  | Labels: (1487,)
Saved to: /kaggle/working/processed_data
Số lượng Features: 70 (34 X,Y + 34 Vel + 1 Hip_H + 1 Angle)
Masking padding value: -99.0
